In [1]:
# ============================================================
# Toronto Restaurant Risk & Compliance Intelligence System
# Notebook 02: Data Cleaning & Transformation
# Author: Sreekar Koduru
# Date: June 2026
# ============================================================

import pandas as pd
import numpy as np
import os
from datetime import datetime

# Paths
RAW_PATH = os.path.expanduser(
    '~/Projects/project3-toronto-restaurant-compliance/data/raw/Dinesafe.csv'
)
CLEANED_PATH = os.path.expanduser(
    '~/Projects/project3-toronto-restaurant-compliance/data/cleaned/dinesafe_cleaned.csv'
)

# Load raw data
df = pd.read_csv(RAW_PATH)

print(f"Raw dataset loaded ✅")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Raw dataset loaded ✅
Rows: 102,766
Columns: 18


In [2]:
# ============================================================
# STEP 1: Standardise Column Names
# Why: Column names are inconsistent — some camelCase (estId),
# some lowercase (address), some TitleCase (OutcomeDate).
# Standardising to snake_case makes all future code consistent
# and prevents errors when referencing columns.
# ============================================================

# Store original columns for reference
original_columns = df.columns.tolist()

# Rename to snake_case
df.columns = [
    'row_id', 'unique_id', 'est_id', 'old_est_id', 'est_name',
    'address', 'inspection_status', 'phone', 'inspection_date',
    'observation', 'type_desc', 'deficiency_desc', 'severity',
    'outcome_date', 'outcome_desc', 'amount_fined',
    'latitude', 'longitude'
]

print("Column names standardised ✅")
print("\nBefore → After:")
for old, new in zip(original_columns, df.columns):
    changed = "← changed" if old != new else ""
    print(f"  {old:<25} → {new:<25} {changed}")

Column names standardised ✅

Before → After:
  _id                       → row_id                    ← changed
  unique_id                 → unique_id                 
  estId                     → est_id                    ← changed
  oldEstId                  → old_est_id                ← changed
  estName                   → est_name                  ← changed
  address                   → address                   
  inspectionStatus          → inspection_status         ← changed
  phone                     → phone                     
  inspectionDate            → inspection_date           ← changed
  observation               → observation               
  typeDesc                  → type_desc                 ← changed
  deficiencyDesc            → deficiency_desc           ← changed
  severity                  → severity                  
  OutcomeDate               → outcome_date              ← changed
  OutcomeDesc               → outcome_desc              ← changed
  amountFi

In [3]:
# ============================================================
# STEP 2: Date Conversion & Time Feature Extraction
# Why: Dates are stored as strings in raw CSV. Python cannot
# do any date math (calculate days, group by month, compare
# years) until they are converted to proper datetime objects.
# We also extract year and month as separate columns because
# SQL and Power BI work better with integer year/month fields
# than with full datetime strings.
# ============================================================

# Convert inspection_date to datetime
df['inspection_date'] = pd.to_datetime(df['inspection_date'], errors='coerce')

# Convert outcome_date to datetime
df['outcome_date'] = pd.to_datetime(df['outcome_date'], errors='coerce')

# Extract time features
df['inspection_year']  = df['inspection_date'].dt.year
df['inspection_month'] = df['inspection_date'].dt.month
df['inspection_month_name'] = df['inspection_date'].dt.strftime('%B')

# Calculate days since inspection
today = pd.Timestamp(datetime.today().date())
df['days_since_inspection'] = (today - df['inspection_date']).dt.days

# Flag active establishments
# Active = inspected within last 24 months (730 days)
df['is_active'] = df['days_since_inspection'] <= 730

print("Date conversion complete ✅")
print(f"\nInspection date range:")
print(f"  Earliest: {df['inspection_date'].min().date()}")
print(f"  Latest:   {df['inspection_date'].max().date()}")
print(f"\nNew columns added:")
print(f"  inspection_year, inspection_month, inspection_month_name")
print(f"  days_since_inspection, is_active")
print(f"\nActive establishments (inspected within 24 months):")
print(f"  Active:   {df['is_active'].sum():,}")
print(f"  Inactive: {(~df['is_active']).sum():,}")

Date conversion complete ✅

Inspection date range:
  Earliest: 2023-11-10
  Latest:   2026-06-20

New columns added:
  inspection_year, inspection_month, inspection_month_name
  days_since_inspection, is_active

Active establishments (inspected within 24 months):
  Active:   81,670
  Inactive: 21,096


In [4]:
# ============================================================
# STEP 3: Severity Standardisation & Numeric Encoding
# Why: The severity field contains text values (M - Minor,
# S - Significant, C - Crucial). For the risk scoring model
# we need numbers, not text. We create a severity_code column
# (0, 1, 2, 3) so we can do mathematical calculations.
# We also standardise the text values because older records
# may have inconsistent formatting.
# ============================================================

# Check current severity values before cleaning
print("Severity values BEFORE cleaning:")
print(df['severity'].value_counts(dropna=False))

# Standardise severity text values
severity_map = {
    'M - Minor'       : 'M - Minor',
    'S - Significant' : 'S - Significant',
    'C - Crucial'     : 'C - Crucial',
}
df['severity'] = df['severity'].map(severity_map)

# Create numeric severity code
# 0 = No violation (null severity = clean inspection)
# 1 = Minor
# 2 = Significant
# 3 = Crucial
severity_code_map = {
    'M - Minor'       : 1,
    'S - Significant' : 2,
    'C - Crucial'     : 3,
}
df['severity_code'] = df['severity'].map(severity_code_map).fillna(0).astype(int)

print("\nSeverity values AFTER cleaning:")
print(df['severity'].value_counts(dropna=False))

print("\nSeverity code distribution:")
print(df['severity_code'].value_counts().sort_index())

print("\nSeverity encoding:")
print("  0 = No violation (clean inspection)")
print("  1 = Minor")
print("  2 = Significant")
print("  3 = Crucial")

Severity values BEFORE cleaning:
severity
NaN                42076
M - Minor          37338
S - Significant    20527
C - Crucial         2825
Name: count, dtype: int64

Severity values AFTER cleaning:
severity
NaN                42076
M - Minor          37338
S - Significant    20527
C - Crucial         2825
Name: count, dtype: int64

Severity code distribution:
severity_code
0    42076
1    37338
2    20527
3     2825
Name: count, dtype: int64

Severity encoding:
  0 = No violation (clean inspection)
  1 = Minor
  2 = Significant
  3 = Crucial


In [5]:
# ============================================================
# STEP 4: Clean Address & Establishment Name
# Why: The address field contains the literal word "None"
# where no unit number exists (e.g. "1871 O'Connor Dr None
# M4A 1X1"). This looks unprofessional in dashboard tables.
# We also standardise establishment names to Title Case
# because the raw data mixes ALL CAPS and mixed case.
# ============================================================

# Check address issue
print("Sample addresses BEFORE cleaning:")
print(df['address'].head(5).tolist())

# Remove " None" from addresses
df['address'] = df['address'].str.replace(' None ', ' ', regex=False)
df['address'] = df['address'].str.strip()

# Standardise establishment names to Title Case
df['est_name'] = df['est_name'].str.title().str.strip()

# Clean type_desc — standardise to Title Case
# Raw data mixes ALL CAPS and sentence case for same violations
df['type_desc'] = df['type_desc'].str.title().str.strip()

# Clean deficiency_desc — standardise to Title Case
df['deficiency_desc'] = df['deficiency_desc'].str.title().str.strip()

print("\nSample addresses AFTER cleaning:")
print(df['address'].head(5).tolist())

print("\nSample establishment names AFTER cleaning:")
print(df['est_name'].head(5).tolist())

print("\nAddress cleaning complete ✅")
print("Establishment name cleaning complete ✅")
print("Type description cleaning complete ✅")

Sample addresses BEFORE cleaning:
["1871 O'Connor Dr None M4A 1X1", "1871 O'Connor Dr None M4A 1X1", "1871 O'Connor Dr None M4A 1X1", "1871 O'Connor Dr None M4A 1X1", "1871 O'Connor Dr None M4A 1X1"]

Sample addresses AFTER cleaning:
["1871 O'Connor Dr M4A 1X1", "1871 O'Connor Dr M4A 1X1", "1871 O'Connor Dr M4A 1X1", "1871 O'Connor Dr M4A 1X1", "1871 O'Connor Dr M4A 1X1"]

Sample establishment names AFTER cleaning:
['# Hashtag India Restaurant', '# Hashtag India Restaurant', '# Hashtag India Restaurant', '# Hashtag India Restaurant', '# Hashtag India Restaurant']

Address cleaning complete ✅
Establishment name cleaning complete ✅
Type description cleaning complete ✅


In [6]:
# ============================================================
# STEP 5: Handle Fine Amounts & Drop Unused Columns
# Why: amount_fined has 99.8% nulls. For aggregation purposes
# we fill nulls with 0 — meaning no fine was issued.
# We also drop columns not used in any analysis to keep the
# cleaned dataset lean and focused.
# Dropped columns:
# - row_id: internal portal ID, not meaningful
# - phone: not used in any analysis
# - observation: standardised text, no analytical value
# - old_est_id: only needed for historical joins (separate process)
# ============================================================

# Fill null fine amounts with 0
df['amount_fined'] = df['amount_fined'].fillna(0)

print("Fine amount nulls filled with 0 ✅")
print(f"Fine amount stats:")
print(f"  Records with fines (>$0): {(df['amount_fined'] > 0).sum():,}")
print(f"  Records with no fine ($0): {(df['amount_fined'] == 0).sum():,}")
print(f"  Total fines issued: ${df['amount_fined'].sum():,.2f}")

# Drop unused columns
cols_to_drop = ['row_id', 'phone', 'observation', 'old_est_id']
df = df.drop(columns=cols_to_drop)

print(f"\nDropped columns: {cols_to_drop}")
print(f"\nRemaining columns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

Fine amount nulls filled with 0 ✅
Fine amount stats:
  Records with fines (>$0): 171
  Records with no fine ($0): 102,595
  Total fines issued: $121,650.00

Dropped columns: ['row_id', 'phone', 'observation', 'old_est_id']

Remaining columns (20):
  - unique_id
  - est_id
  - est_name
  - address
  - inspection_status
  - inspection_date
  - type_desc
  - deficiency_desc
  - severity
  - outcome_date
  - outcome_desc
  - amount_fined
  - latitude
  - longitude
  - inspection_year
  - inspection_month
  - inspection_month_name
  - days_since_inspection
  - is_active
  - severity_code


In [7]:
# ============================================================
# STEP 6: Final Validation & Save Cleaned Dataset
# Why: Before saving, we do a final check to confirm the
# cleaned dataset is complete and correct. We compare
# row counts between raw and cleaned to ensure no records
# were accidentally dropped. Then we save to the cleaned
# folder — this is the file all subsequent analysis
# (MySQL, Power BI, Excel) will use.
# ============================================================

print("=" * 60)
print("CLEANED DATASET SUMMARY")
print("=" * 60)

print(f"\nRow count comparison:")
print(f"  Raw dataset:     102,766")
print(f"  Cleaned dataset: {len(df):,}")
print(f"  Difference:      {102766 - len(df):,} (should be 0)")

print(f"\nColumn count:")
print(f"  Raw:     18 columns")
print(f"  Cleaned: {len(df.columns)} columns")
print(f"  Added:   5 derived columns")
print(f"  Dropped: 4 unused columns")

print(f"\nData types:")
print(df.dtypes)

print(f"\nActive vs Inactive records:")
print(f"  Active (≤730 days): {df['is_active'].sum():,}")
print(f"  Inactive (>730 days): {(~df['is_active']).sum():,}")

print(f"\nInspection status breakdown:")
for val, count in df['inspection_status'].value_counts().items():
    pct = count/len(df)*100
    print(f"  {val:<20} {count:>7,} ({pct:.1f}%)")

print(f"\nSeverity breakdown:")
for val, count in df['severity_code'].value_counts().sort_index().items():
    labels = {0:'No violation', 1:'Minor', 2:'Significant', 3:'Crucial'}
    pct = count/len(df)*100
    print(f"  {val} — {labels[val]:<15} {count:>7,} ({pct:.1f}%)")

# Save cleaned dataset
os.makedirs(os.path.dirname(CLEANED_PATH), exist_ok=True)
df.to_csv(CLEANED_PATH, index=False)

print(f"\n{'=' * 60}")
print(f"CLEANED DATASET SAVED ✅")
print(f"Location: data/cleaned/dinesafe_cleaned.csv")
print(f"{'=' * 60}")

CLEANED DATASET SUMMARY

Row count comparison:
  Raw dataset:     102,766
  Cleaned dataset: 102,766
  Difference:      0 (should be 0)

Column count:
  Raw:     18 columns
  Cleaned: 20 columns
  Added:   5 derived columns
  Dropped: 4 unused columns

Data types:
unique_id                        object
est_id                           object
est_name                         object
address                          object
inspection_status                object
inspection_date          datetime64[ns]
type_desc                        object
deficiency_desc                  object
severity                         object
outcome_date             datetime64[ns]
outcome_desc                     object
amount_fined                    float64
latitude                        float64
longitude                       float64
inspection_year                   int32
inspection_month                  int32
inspection_month_name            object
days_since_inspection             int64
is_active      